# **1. Notebook Setup**

## **1.1 Imports**

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import yfinance as yf


## **1.2 Notebook Configuration**

In [2]:
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

## **1.3 Paths**

In [3]:
PROJECT_ROOT = Path().resolve().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
INTERIM_DATA_DIR = DATA_DIR / "interim"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"

SRC_DIR = PROJECT_ROOT / "src"

# Add project root to Python path
sys.path.append(str(PROJECT_ROOT))

# **2. Data Pull and Processing**

In [4]:
from src.yfinance_loader import fetch_multiple_companies

### **2.1 Sample pull**

In [5]:
TICKERS = [
    "AAPL",
    "MSFT",
    "GOOGL",
    "AMZN",
    "META"
]

SLEEP_SECONDS = 1

In [6]:
fetch_log = fetch_multiple_companies(
    tickers=TICKERS,
    output_dir=RAW_DATA_DIR / "yfinance",
    sleep_seconds=SLEEP_SECONDS
)

fetch_log.head()

Fetching AAPL...


Fetching MSFT...


Fetching GOOGL...


Fetching AMZN...


Fetching META...


,ticker,status,error
0,AAPL,success,None
1,MSFT,success,None
2,GOOGL,success,None
3,AMZN,success,None
4,META,success,None


Everything looks fine for the sample list

### **2.2 Pull Universe of Tickers**

In [7]:
import signal

class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException()

In [8]:
from src.data.ticker_universe import build_us_equity_universe

us_equity_universe = build_us_equity_universe(
    remove_non_operating_securities=True,
    output_path=RAW_DATA_DIR / "ticker_universe" / "us_equity_universe.xlsx",
)

us_equity_universe.head(10)

,Symbol,Company Name,Exchange
0,ACCS,ACCESS Newswire Inc. Common Stock,AMEX
1,AEON,"AEON Biopharma, Inc. Class A Common Stock",AMEX
2,AGIG,Abundia Global Impact Group Inc. Common stock,AMEX
3,AIB,"BlockchAIn Digital Infrastructure, Inc Common ...",AMEX
4,AIRI,Air Industries Group Common Stock,AMEX
5,AMBO,Ambow Education Holding Ltd. American Deposito...,AMEX
6,AMS,American Shared Hospital Services Common Stock,AMEX
7,AMZE,"Amaze Holdings, Inc. Common Stock",AMEX
8,APT,"Alpha Pro Tech, Ltd. Common Stock",AMEX
9,APUS,"Apimeds Pharmaceuticals US, Inc. Common Stock",AMEX


In [9]:
us_equity_universe["Exchange"].value_counts()

Exchange
NASDAQ    3307
NYSE      1857
AMEX       239
Name: count, dtype: int64

Number of company tickers are within the ballpark. Good to preceed

### **2.3 Pull Financials based on Ticker Universe**

In [10]:
us_equity_universe = us_equity_universe[
    ~us_equity_universe["Symbol"].str.contains(r"[.$^]", regex=True, na=False)
].copy()

from src.data.financials_loader import (
    fetch_and_flatten_financials,
    filter_companies_with_positive_revenue,
)

### **2.4 Flatten file**

In [11]:
financials_flat_df, financials_fetch_log = fetch_and_flatten_financials(
    ticker_universe=us_equity_universe,
    output_dir=RAW_DATA_DIR / "yfinance_financials",
    sleep_seconds=0.75,
    save_every=100,
)

financials_flat_df.head()

Fetching financial statements:   0%|          | 0/5354 [00:00<?, ?it/s]

Processed 100 of 5,354 companies...


Processed 200 of 5,354 companies...


Processed 300 of 5,354 companies...


Processed 400 of 5,354 companies...


...

Processed 5,100 of 5,354 companies...


Processed 5,200 of 5,354 companies...


Processed 5,300 of 5,354 companies...


Processed 5,400 of 5,354 companies...


,Symbol,Company Name,Exchange,Income_Tax Effect Of Unusual Items,Income_Tax Rate For Calcs,Income_Normalized EBITDA,Income_Total Unusual Items,Income_Total Unusual Items Excluding Goodwill,Income_Net Income From Continuing Operation Net Minority Interest,Income_Reconciled Depreciation,Income_Reconciled Cost Of Revenue,Income_EBITDA,Income_EBIT,Income_Net Interest Income,Income_Interest Expense,Income_Interest Income,Income_Normalized Income,Income_Net Income From Continuing And Discontinued Operation,Income_Total Expenses,Income_Total Operating Income As Reported,Income_Diluted Average Shares,Income_Basic Average Shares,Income_Diluted EPS,Income_Basic EPS,Income_Diluted NI Availto Com Stockholders,Income_Net Income Common Stockholders,Income_Net Income,Income_Net Income Including Noncontrolling Interests,Income_Net Income Discontinuous Operations,Income_Net Income Continuous Operations,Income_Tax Provision,Income_Pretax Income,Income_Other Income Expense,Income_Other Non Operating Income Expenses,Income_Special Income Charges,Income_Other Special Charges,Income_Impairment Of Capital Assets,Income_Gain On Sale Of Security,Income_Net Non Operating Interest Income Expense,Income_Total Other Finance Cost,Income_Interest Expense Non Operating,Income_Interest Income Non Operating,Income_Operating Income,Income_Operating Expense,Income_Depreciation Amortization Depletion Income Statement,Income_Depreciation And Amortization In Income Statement,Income_Research And Development,Income_Selling General And Administration,Income_Selling And Marketing Expense,Income_General And Administrative Expense,Income_Other Gand A,Income_Gross Profit,Income_Cost Of Revenue,Income_Total Revenue,Income_Operating Revenue,Balance_Treasury Shares Number,Balance_Ordinary Shares Number,Balance_Share Issued,Balance_Net Debt,Balance_Total Debt,Balance_Tangible Book Value,Balance_Invested Capital,Balance_Working Capital,Balance_Net Tangible Assets,Balance_Capital Lease Obligations,Balance_Common Stock Equity,Balance_Total Capitalization,Balance_Total Equity Gross Minority Interest,Balance_Stockholders Equity,Balance_Gains Losses Not Affecting Retained Earnings,Balance_Other Equity Adjustments,Balance_Retained Earnings,Balance_Additional Paid In Capital,Balance_Capital Stock,Balance_Common Stock,Balance_Preferred Stock,Balance_Total Liabilities Net Minority Interest,Balance_Total Non Current Liabilities Net Minority Interest,Balance_Other Non Current Liabilities,Balance_Non Current Deferred Liabilities,Balance_Non Current Deferred Taxes Liabilities,Balance_Long Term Debt And Capital Lease Obligation,Balance_Long Term Capital Lease Obligation,Balance_Long Term Debt,Balance_Current Liabilities,Balance_Other Current Liabilities,Balance_Current Deferred Liabilities,Balance_Current Deferred Revenue,Balance_Current Debt And Capital Lease Obligation,Balance_Current Debt,Balance_Other Current Borrowings,Balance_Current Notes Payable,Balance_Payables And Accrued Expenses,Balance_Current Accrued Expenses,Balance_Payables,Balance_Total Tax Payable,Balance_Income Tax Payable,Balance_Accounts Payable,Balance_Total Assets,Balance_Total Non Current Assets,Balance_Other Non Current Assets,Balance_Non Current Deferred Assets,Balance_Non Current Deferred Taxes Assets,Balance_Goodwill And Other Intangible Assets,Balance_Other Intangible Assets,Balance_Goodwill,Balance_Net PPE,Balance_Accumulated Depreciation,Balance_Gross PPE,Balance_Leases,Balance_Other Properties,Balance_Machinery Furniture Equipment,Balance_Properties,Balance_Current Assets,Balance_Other Current Assets,Balance_Assets Held For Sale Current,Balance_Receivables,Balance_Taxes Receivable,Balance_Accounts Receivable,Balance_Allowance For Doubtful Accounts Receivable,Balance_Gross Accounts Receivable,Balance_Cash Cash Equivalents And Short Term Investments,Balance_Cash And Cash Equivalents,CashFlow_Free Cash Flow,CashFlow_Repurchase Of Capital Stock,CashFlow_Repayment Of Debt,CashFlow_Issuance Of Debt,CashFlow_Capital

In [12]:
financials_fetch_log["Status"].value_counts()

Status
Success    5354
Name: count, dtype: int64

### **2.5 Filter and Save**

Here we filter out the companies that have revenue < 0, as most KPI calculations require a valid-positive revenue figure

In [14]:
filtered_financials_df = filter_companies_with_positive_revenue(
    financials_df=financials_flat_df,
    revenue_column="Income_Total Revenue",
)

filtered_financials_df.to_excel(
    INTERIM_DATA_DIR / "financials_positive_revenue.xlsx",
    index=False
)

print(f"Original companies: {len(financials_flat_df):,}")
print(f"Companies with positive revenue: {len(filtered_financials_df):,}")

filtered_financials_df.head()

Original companies: 5,354
Companies with positive revenue: 4,576


,Symbol,Company Name,Exchange,Income_Tax Effect Of Unusual Items,Income_Tax Rate For Calcs,Income_Normalized EBITDA,Income_Total Unusual Items,Income_Total Unusual Items Excluding Goodwill,Income_Net Income From Continuing Operation Net Minority Interest,Income_Reconciled Depreciation,Income_Reconciled Cost Of Revenue,Income_EBITDA,Income_EBIT,Income_Net Interest Income,Income_Interest Expense,Income_Interest Income,Income_Normalized Income,Income_Net Income From Continuing And Discontinued Operation,Income_Total Expenses,Income_Total Operating Income As Reported,Income_Diluted Average Shares,Income_Basic Average Shares,Income_Diluted EPS,Income_Basic EPS,Income_Diluted NI Availto Com Stockholders,Income_Net Income Common Stockholders,Income_Net Income,Income_Net Income Including Noncontrolling Interests,Income_Net Income Discontinuous Operations,Income_Net Income Continuous Operations,Income_Tax Provision,Income_Pretax Income,Income_Other Income Expense,Income_Other Non Operating Income Expenses,Income_Special Income Charges,Income_Other Special Charges,Income_Impairment Of Capital Assets,Income_Gain On Sale Of Security,Income_Net Non Operating Interest Income Expense,Income_Total Other Finance Cost,Income_Interest Expense Non Operating,Income_Interest Income Non Operating,Income_Operating Income,Income_Operating Expense,Income_Depreciation Amortization Depletion Income Statement,Income_Depreciation And Amortization In Income Statement,Income_Research And Development,Income_Selling General And Administration,Income_Selling And Marketing Expense,Income_General And Administrative Expense,Income_Other Gand A,Income_Gross Profit,Income_Cost Of Revenue,Income_Total Revenue,Income_Operating Revenue,Balance_Treasury Shares Number,Balance_Ordinary Shares Number,Balance_Share Issued,Balance_Net Debt,Balance_Total Debt,Balance_Tangible Book Value,Balance_Invested Capital,Balance_Working Capital,Balance_Net Tangible Assets,Balance_Capital Lease Obligations,Balance_Common Stock Equity,Balance_Total Capitalization,Balance_Total Equity Gross Minority Interest,Balance_Stockholders Equity,Balance_Gains Losses Not Affecting Retained Earnings,Balance_Other Equity Adjustments,Balance_Retained Earnings,Balance_Additional Paid In Capital,Balance_Capital Stock,Balance_Common Stock,Balance_Preferred Stock,Balance_Total Liabilities Net Minority Interest,Balance_Total Non Current Liabilities Net Minority Interest,Balance_Other Non Current Liabilities,Balance_Non Current Deferred Liabilities,Balance_Non Current Deferred Taxes Liabilities,Balance_Long Term Debt And Capital Lease Obligation,Balance_Long Term Capital Lease Obligation,Balance_Long Term Debt,Balance_Current Liabilities,Balance_Other Current Liabilities,Balance_Current Deferred Liabilities,Balance_Current Deferred Revenue,Balance_Current Debt And Capital Lease Obligation,Balance_Current Debt,Balance_Other Current Borrowings,Balance_Current Notes Payable,Balance_Payables And Accrued Expenses,Balance_Current Accrued Expenses,Balance_Payables,Balance_Total Tax Payable,Balance_Income Tax Payable,Balance_Accounts Payable,Balance_Total Assets,Balance_Total Non Current Assets,Balance_Other Non Current Assets,Balance_Non Current Deferred Assets,Balance_Non Current Deferred Taxes Assets,Balance_Goodwill And Other Intangible Assets,Balance_Other Intangible Assets,Balance_Goodwill,Balance_Net PPE,Balance_Accumulated Depreciation,Balance_Gross PPE,Balance_Leases,Balance_Other Properties,Balance_Machinery Furniture Equipment,Balance_Properties,Balance_Current Assets,Balance_Other Current Assets,Balance_Assets Held For Sale Current,Balance_Receivables,Balance_Taxes Receivable,Balance_Accounts Receivable,Balance_Allowance For Doubtful Accounts Receivable,Balance_Gross Accounts Receivable,Balance_Cash Cash Equivalents And Short Term Investments,Balance_Cash And Cash Equivalents,CashFlow_Free Cash Flow,CashFlow_Repurchase Of Capital Stock,CashFlow_Repayment Of Debt,CashFlow_Issuance Of Debt,CashFlow_Capital

# **3. Final Data Check**

### **3.1 Data Quality**

In [15]:
print(financials_flat_df.shape)
print(filtered_financials_df.shape)

(5354, 351)
(4576, 351)


In [16]:
filtered_financials_df[[
    "Income_Total Revenue"
]].describe()

,Income_Total Revenue
count,4.576000e+03
mean,1.800574e+11
std,3.734056e+12
min,5.500000e+02
25%,6.852400e+07
50%,5.943835e+08
75%,3.379406e+09
max,1.467420e+14


In [18]:
filtered_financials_df.isna().mean().sort_values(ascending=False).head(50)

CashFlow_Dividends Paid Direct                                                  1.000000
Balance_Restricted Common Stock                                                 0.999781
CashFlow_Receiptsfrom Government Grants                                         0.999563
CashFlow_Dividends Received Direct                                              0.999344
Income_Depletion Income Statement                                               0.999126
CashFlow_Paymentson Behalfof Employees                                          0.998907
CashFlow_Change In Dividend Payable                                             0.998689
Balance_General Partnership Capital                                             0.998470
CashFlow_Dividend Paid Cfo                                                      0.998252
Income_Net Income From Tax Loss Carryforward                                    0.998252
CashFlow_Other Cash Paymentsfrom Operating Activities                           0.998033
CashFlow_Other Cash R

### **3.2 Failure logs**

In [19]:
failed_fetches_df = financials_fetch_log[
    financials_fetch_log["Status"] != "Success"
]

failed_fetches_df.to_excel(
    OUTPUTS_DIR / "failed_yfinance_fetches.xlsx",
    index=False
)

print(f"Failed fetches: {len(failed_fetches_df):,}")

Failed fetches: 0


# Notebook Summary

This notebook constructed the initial U.S. equity universe by combining NASDAQ, NYSE, and AMEX listings from NasdaqTrader. The dataset was cleaned to remove ETFs, test issues, and likely non-operating securities.

Financial statement data was then retrieved from Yahoo Finance using the yfinance library. Income statements, balance sheets, and cash flow statements were flattened into a tabular structure suitable for downstream KPI engineering and machine learning workflows.

Checkpoint saving, logging, and validation procedures were implemented to improve robustness and reproducibility. Finally, companies without valid positive revenue values were filtered out to improve dataset quality for subsequent clustering analysis.